# 📊 什麼因素造就一部熱門電視劇？
### 學生資料實驗室（IMDb + Netflix + TMDb）

你將分析真實的電視劇資料，找出讓劇集受歡迎的原因。

In [1]:
# !make init
# !python -m netflix.fetch
# !python -m netflix.build

In [2]:
import pandas as pd

imdb = pd.read_csv("netflix/data/imdb.titles.composite.csv")
netflix = pd.read_csv("netflix/data/netflix.titles.composite.csv")
tmdb = pd.read_csv("netflix/data/tmdb.titles.v3.csv")

print(imdb.shape, netflix.shape, tmdb.shape)

(299097, 15) (691, 13) (168639, 29)


In [3]:
print("imdb", imdb.shape)
imdb.head(3)

imdb (299097, 15)


,tconst,titleType,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,genres,title_key,averageRating,numVotes,all_akas,cast,directors,writers
0,tt0030138,tvSeries,Flash Gordon's Trip to Mars,Flash Gordon's Trip to Mars,1938,\N,299,"Action,Adventure,Family",lash ordons rip to ars,6.9,1156.0,flash gordon's trip to mars: chapter four - an...,Kenne Duncan | Charles Middleton | C. Montague...,"nm0066247,nm0384616,nm0826873,nm0853028","nm0198143,nm0321216,nm0355963,nm0713204,nm0870..."
1,tt0035803,tvSeries,The German Weekly Review,Die Deutsche Wochenschau,1940,1945,\N,"Documentary,News",he erman eekly eview,8.2,72.0,die deutsche wochenschau | images d'actualité ...,Joseph Goebbels | Hermann Göring | Hans Frank ...,\N,\N
2,tt0039123,tvSeries,Kraft Mystery Theatre,Kraft Television Theatre,1947,1958,60,Drama,raft ystery heatre,7.8,247.0,kraft television theatre | kraft mystery theat...,Mercer McLeod | Vaughn Taylor | E.G. Marshall ...,"nm0390776,nm2427430,nm0334353,nm0242409,nm1802...","nm1055756,nm0590316,nm0913670,nm0044801,nm0442..."


In [4]:
print("tmdb", tmdb.shape)
tmdb.head(3)

tmdb (168639, 29)


,id,name,number_of_seasons,number_of_episodes,original_language,vote_count,vote_average,overview,adult,backdrop_path,...,tagline,genres,created_by,languages,networks,origin_country,spoken_languages,production_companies,production_countries,episode_run_time
0,1399,Game of Thrones,8,73,en,21857,8.442,Seven noble families fight for control of the ...,False,/2OMB0ynKlyIenMJWI2Dy9IWT4c.jpg,...,Winter Is Coming,"Sci-Fi & Fantasy, Drama, Action & Adventure","David Benioff, D.B. Weiss",en,HBO,US,English,"Revolution Sun Studios, Television 360, Genera...","United Kingdom, United States of America",0
1,71446,Money Heist,3,41,es,17836,8.257,"To carry out the biggest heist in history, a m...",False,/gFZriCkpJYsApPZEF3jhxL4yLzG.jpg,...,The perfect robbery.,"Crime, Drama",Álex Pina,es,"Netflix, Antena 3",ES,Español,Vancouver Media,Spain,70
2,66732,Stranger Things,4,34,en,16161,8.624,"When a young boy vanishes, a small town uncove...",False,/2MaumbgBlW1NoPo3ZJO38A6v7OS.jpg,...,Every ending has a beginning.,"Drama, Sci-Fi & Fantasy, Mystery","Matt Duffer, Ross Duffer",en,Netflix,US,English,"21 Laps Entertainment, Monkey Massacre Product...",United States of America,0


In [5]:
print("netflix", netflix.shape)
netflix.head(3)

netflix (691, 13)


,key,netflix_viewing_hours,netflix_weeks,netflix_year_hint,netflix_title,netflix_clean_title,tmdb_title,tmdb_popularity,tmdb_vote_average,tmdb_vote_count,imdb_title,imdb_averageRating,imdb_numVotes
0,10,26580000,2,2021,The 100,the 100,The 100,127.224,7.916,7666.0,The 100!,NaN,NaN
1,10_fr_mi,13590000,2,2021,1000 Miles from Christmas,1000 miles from christmas,100 Miles From Nowhere,1.400,3.600,5.0,1000 Miles from Christmas,5.9,2334.0
2,10_lo,12600000,1,2021,Love 101,love 101,Love 101,14.518,7.974,154.0,Love 101,7.4,20069.0


In [6]:
imdb_extra = (
    imdb[["primaryTitle", "genres", "runtimeMinutes", "cast", "directors", "writers"]]
    .dropna(subset=["primaryTitle"])
    .drop_duplicates(subset="primaryTitle", keep="first")
    .rename(columns={
        "primaryTitle": "imdb_title",
        "genres": "imdb_genres",
        "runtimeMinutes": "imdb_runtimeMinutes",
        "cast": "imdb_cast",
        "directors": "imdb_directors",
        "writers": "imdb_writers",
    })
)

tmdb_extra = (
    tmdb[["name", "number_of_seasons", "number_of_episodes", "original_language", "networks"]]
    .dropna(subset=["name"])
    .drop_duplicates(subset="name", keep="first")
    .rename(columns={
        "name": "tmdb_title",
        "number_of_seasons": "tmdb_number_of_seasons",
        "number_of_episodes": "tmdb_number_of_episodes",
        "original_language": "tmdb_original_language",
        "networks": "tmdb_networks",
    })
)

df = (
    netflix
    .merge(imdb_extra, on="imdb_title", how="left")
    .merge(tmdb_extra, on="tmdb_title", how="left")
)

print("merged df shape:", df.shape)
df.head(5)

merged df shape: (691, 22)


,key,netflix_viewing_hours,netflix_weeks,netflix_year_hint,netflix_title,netflix_clean_title,tmdb_title,tmdb_popularity,tmdb_vote_average,tmdb_vote_count,...,imdb_numVotes,imdb_genres,imdb_runtimeMinutes,imdb_cast,imdb_directors,imdb_writers,tmdb_number_of_seasons,tmdb_number_of_episodes,tmdb_original_language,tmdb_networks
0,10,26580000,2,2021,The 100,the 100,The 100,127.224,7.916,7666.0,...,NaN,NaN,NaN,NaN,NaN,NaN,7.0,100.0,en,The CW
1,10_fr_mi,13590000,2,2021,1000 Miles from Christmas,1000 miles from christmas,100 Miles From Nowhere,1.400,3.600,5.0,...,2334.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0,en,Animal Planet
2,10_lo,12600000,1,2021,Love 101,love 101,Love 101,14.518,7.974,154.0,...,20069.0,"Comedy,Drama,Romance",45,Mert Yazicioglu | Alina Boz | Kubilay Aka | Mü...,"nm3446789,nm5519375,nm2141029,nm3828643",nm1689260,2.0,16.0,tr,Netflix
3,12_st,16420000,2,2022,12 Strong,12 strong,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,13_ho_se,26250000,2,2022,13 Hours: The Secret Soldiers of Benghazi,13 hours the secret soldiers of benghazi,NaN,NaN,NaN,NaN,...,72.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 檢查資料匹配覆蓋率

In [7]:
import numpy as np
import pandas as pd

def match_coverage(frame: pd.DataFrame, source_columns: dict[str, str]) -> pd.DataFrame:
    """回報各資料來源中，有多少比例的資料列存在非空匹配。

    Args:
        frame: 合併後的複合資料集。
        source_columns: 將人類可讀的資料來源名稱對應到該來源中
            用來檢查是否為空值的欄位，例如 {"tmdb": "tmdb_title"}。

    Returns:
        一個小型 DataFrame，每個資料來源一列：匹配數、總數，
        以及匹配覆蓋率（百分比）。
    """
    rows = []
    total = len(frame)
    for source_name, column in source_columns.items():
        matched = frame[column].notna().sum()
        rows.append({"source": source_name, "matched": matched, "total": total,
                      "coverage_pct": round(100 * matched / total, 1)})
    return pd.DataFrame(rows)


match_coverage(df, {"tmdb": "tmdb_title", "imdb": "imdb_title"})

,source,matched,total,coverage_pct
0,tmdb,394,691,57.0
1,imdb,665,691,96.2


## 定義目標：怎樣才算「熱門」？

### 候選的熱門訊號彼此相關性如何？

In [8]:
# 驗證 label_hits() 的設計：為什麼用 3 個資料來源、
# 為什麼 min_sources=2，為什麼 quantile=0.75？
hit_source_cols = ["netflix_viewing_hours", "imdb_numVotes", "tmdb_popularity"]

# 1) 缺值狀況：如果要求 3 個來源都同時存在資料，
#    到底有多少列會符合資格？
print("=== 各資料來源缺值比例 ===")
print(df[hit_source_cols].isna().mean().round(3))
print()
print("三個來源皆存在:", df[hit_source_cols].notna().all(axis=1).sum(), "/", len(df))
print("至少兩個來源存在:", (df[hit_source_cols].notna().sum(axis=1) >= 2).sum(), "/", len(df))
print()

# 2) 資料來源間的相關性：這些訊號是彼此重複，
#    還是各自捕捉到不同的東西（這能證明跨來源交叉檢查的必要性）？
print("=== 三個熱門訊號來源間的 Spearman 相關係數 ===")
print(df[hit_source_cols].corr(method="spearman").round(2))

=== 各資料來源缺值比例 ===
netflix_viewing_hours    0.000
imdb_numVotes            0.275
tmdb_popularity          0.430
dtype: float64

三個來源皆存在: 279 / 691
至少兩個來源存在: 616 / 691

=== 三個熱門訊號來源間的 Spearman 相關係數 ===
                       netflix_viewing_hours  imdb_numVotes  tmdb_popularity
netflix_viewing_hours                   1.00           0.10             0.38
imdb_numVotes                           0.10           1.00             0.28
tmdb_popularity                         0.38           0.28             1.00


In [9]:
def label_hits(frame, source_cols=("netflix_viewing_hours", "imdb_numVotes", "tmdb_popularity"),
               quantile=0.75, min_sources=2):
    votes = pd.DataFrame({
        col: frame[col] >= frame[col].quantile(quantile) for col in source_cols
    })
    return votes.sum(axis=1) >= min_sources

df["is_hit"] = label_hits(df)
df["is_hit"].value_counts()

is_hit
False    601
True      90
Name: count, dtype: int64

## 追劇速度：上榜期間每週觀看時數

In [10]:
def binge_velocity(frame: pd.DataFrame,
                    hours_col: str = "netflix_viewing_hours",
                    weeks_col: str = "netflix_weeks") -> pd.Series:
    """上榜的 Top 10 清單期間，平均每週觀看時數。

    避免任何 weeks_col == 0 的劇集發生除以零的錯誤。
    """
    weeks = frame[weeks_col].replace(0, np.nan)
    return (frame[hours_col] / weeks).rename("binge_velocity")


df["binge_velocity"] = binge_velocity(df)
df[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "binge_velocity"]].head(3)

,netflix_title,netflix_viewing_hours,netflix_weeks,binge_velocity
0,The 100,26580000,2,13290000.0
1,1000 Miles from Christmas,13590000,2,6795000.0
2,Love 101,12600000,1,12600000.0


## 更可信的評分：把小樣本向平均值收縮

In [11]:
def bayesian_rating(frame: pd.DataFrame,
                     rating_col: str,
                     votes_col: str,
                     prior_votes: int = 500) -> pd.Series:
    """依票數多寡，把每個評分向資料集平均值收縮。

    weighted_rating = (v / (v + m)) * R + (m / (v + m)) * C

    其中 R 是該劇集自己的評分，v 是它的票數，C 是全資料集的
    平均評分，m 則是 `prior_votes` 這個「信心」門檻：票數
    v << m 的劇集會被大力拉向 C，票數 v >> m 的劇集則幾乎不變。
    """
    v = frame[votes_col]
    R = frame[rating_col]
    C = frame[rating_col].mean()
    m = prior_votes
    return ((v / (v + m)) * R + (m / (v + m)) * C).rename(f"{rating_col}_shrunk")


df["imdb_rating_shrunk"] = bayesian_rating(df, "imdb_averageRating", "imdb_numVotes")
df[["netflix_title", "imdb_averageRating", "imdb_numVotes", "imdb_rating_shrunk"]].dropna().head(3)

,netflix_title,imdb_averageRating,imdb_numVotes,imdb_rating_shrunk
1,1000 Miles from Christmas,5.9,2334.0,6.011140
2,Love 101,7.4,20069.0,7.378850
4,13 Hours: The Secret Soldiers of Benghazi,6.6,72.0,6.538759


## 影評與觀眾的評分落差

In [12]:
def audience_alignment_gap(frame: pd.DataFrame,
                            col_a: str = "tmdb_vote_average",
                            col_b: str = "imdb_averageRating") -> pd.Series:
    """兩個評分來源在同一量尺上的絕對差距。"""
    return (frame[col_a] - frame[col_b]).abs().rename("audience_alignment_gap")


df["audience_alignment_gap"] = audience_alignment_gap(df)
df[["netflix_title", "tmdb_vote_average", "imdb_averageRating", "audience_alignment_gap"]].dropna().head(3)

,netflix_title,tmdb_vote_average,imdb_averageRating,audience_alignment_gap
1,1000 Miles from Christmas,3.600,5.9,2.300
2,Love 101,7.974,7.4,0.574
6,Back to 15,6.477,7.0,0.523


## 話題熱度：用對數轉換馴服偏態的票數分佈

In [13]:
def log_buzz(frame: pd.DataFrame, votes_col: str) -> pd.Series:
    """對票數做 log1p 轉換 -- 一種壓縮過的「關注度」特徵。"""
    return np.log1p(frame[votes_col]).rename(f"{votes_col}_log")


df["imdb_buzz_log"] = log_buzz(df, "imdb_numVotes")
df[["netflix_title", "imdb_numVotes", "imdb_buzz_log"]].dropna().head(3)

,netflix_title,imdb_numVotes,imdb_buzz_log
1,1000 Miles from Christmas,2334.0,7.755767
2,Love 101,20069.0,9.906981
4,13 Hours: The Secret Soldiers of Benghazi,72.0,4.290459


## 把所有特徵組裝成一個特徵矩陣

In [14]:
def build_feature_matrix(raw: pd.DataFrame) -> pd.DataFrame:
    """把原始的複合資料集轉換成可直接建模的特徵矩陣。

    串接上面所有的特徵函式。沒有 IMDb 匹配的資料列，其 IMDb
    衍生特徵會保留 NaN -- 後續要選擇丟棄、補值，或明確對這種
    缺值情況建模。
    """
    frame = raw.copy()
    frame["is_hit"] = label_hits(frame)
    frame["binge_velocity"] = binge_velocity(frame)
    frame["imdb_rating_shrunk"] = bayesian_rating(frame, "imdb_averageRating", "imdb_numVotes")
    frame["audience_alignment_gap"] = audience_alignment_gap(frame)
    frame["imdb_buzz_log"] = log_buzz(frame, "imdb_numVotes")

    feature_columns = [
        "binge_velocity",
        "imdb_rating_shrunk",
        "audience_alignment_gap",
        "imdb_buzz_log",
        "tmdb_popularity", 
    ]
    return frame[["netflix_title", "is_hit", *feature_columns]]


features = build_feature_matrix(df)
features.head(10)
# features.describe()

,netflix_title,is_hit,binge_velocity,imdb_rating_shrunk,audience_alignment_gap,imdb_buzz_log,tmdb_popularity
0,The 100,False,13290000.0,NaN,NaN,NaN,127.224
1,1000 Miles from Christmas,False,6795000.0,6.011140,2.300,7.755767,1.400
2,Love 101,False,12600000.0,7.378850,0.574,9.906981,14.518
3,12 Strong,False,8210000.0,NaN,NaN,NaN,NaN
4,13 Hours: The Secret Soldiers of Benghazi,False,13125000.0,6.538759,NaN,4.290459,NaN
5,14 Peaks: Nothing Is Impossible,False,12170000.0,7.682995,NaN,10.431318,NaN
6,Back to 15,False,18060000.0,6.909430,0.523,7.647786,5.340
7,1917,False,10840000.0,6.527022,2.300,4.007333,3.359
8,Chernobyl 1986,False,6274000.0,6.132641,NaN,5.752573,NaN
9,Formula 1: Drive to Survive,False,28515000.0,NaN,NaN,NaN,19.018


## 合理性檢查

In [15]:
numeric = features.drop(columns=["netflix_title"]).astype(float)
numeric.corr()["is_hit"].drop("is_hit").sort_values(key=abs, ascending=False)

correlation_matrix = numeric.corr()
correlation_matrix

,is_hit,binge_velocity,imdb_rating_shrunk,audience_alignment_gap,imdb_buzz_log,tmdb_popularity
is_hit,1.000000,0.493926,0.295843,-0.288071,0.277751,0.312319
binge_velocity,0.493926,1.000000,0.128544,-0.118087,-0.052083,0.187976
imdb_rating_shrunk,0.295843,0.128544,1.000000,-0.253346,0.176088,0.134790
audience_alignment_gap,-0.288071,-0.118087,-0.253346,1.000000,-0.275870,-0.134216
imdb_buzz_log,0.277751,-0.052083,0.176088,-0.275870,1.000000,0.139634
tmdb_popularity,0.312319,0.187976,0.134790,-0.134216,0.139634,1.000000


## 練習 9 — 完美劇集的樣貌

利用資料集中的證據，寫下你心目中理想劇集的配方。

在練習 10 中，我們建立了一個自訂的數學公式 **`hit_score`**，用來評估並排名每部電視劇有多接近我們心目中的「完美劇集樣貌」。以下是這個設計背後一步一步的邏輯：

#### 1. 為什麼需要 `norm(c)`（正規化）？
* **問題所在：** 我們的特徵是用完全不同的尺度去衡量的。例如，`binge_velocity` 是用**數千萬小時**來計算，而 `imdb_rating_shrunk` 則是 **1 到 10** 分的量尺。如果直接把它們加總，龐大的觀看時數會完全淹沒評分的影響力。
* **解決方法：** 我們用 `norm(c)` 把每一個特徵都縮放到 **0 到 1** 之間（0 代表資料集中最差，1 代表最好）。這樣才能做到公平的「同尺度比較」。

#### 2. 為什麼要乘上「權重」（加乘）？
* **原因：** 並非每個特徵都同等重要。我們在練習 9 的相關性分析已經證明，某些因素對劇集是否「爆紅」的影響力遠大於其他因素。
* **運作方式：** 我們依照每個正規化特徵與 `is_hit` 的相關性強弱，給予不同的**權重**：
  * **`binge_velocity`（權重：0.49）：** 獲得最高權重，因為「大家追劇的速度有多快」是成功與否的絕對關鍵。
  * **`audience_alignment_gap`（權重：-0.28）：** 獲得**負**權重，因為落差越大（越兩極化）對劇集越不利。意見分歧會被當作扣分項，把總分往下拉。
  * **其他特徵（TMDb 人氣、IMDb 評分、話題熱度）：** 依其統計影響力，獲得中等程度的正權重。

#### 3. 最終公式
$$\text{Hit Score} = \sum (\text{權重} \times \text{正規化後的特徵})$$

把每個標準化後的特徵乘上它的重要性（權重）再加總起來，我們就得到一個統一的排名指標。分數越高，代表這部劇越接近「完美」！

## 練習 10 — 打造 Hit Score（熱門分數）

In [16]:
def norm(c):
    return (c - c.min()) / (c.max() - c.min())

weights = {
    "binge_velocity":        0.49,
    "tmdb_popularity":       0.31,
    "imdb_rating_shrunk":    0.29,
    "audience_alignment_gap": -0.28,   # 負號：分歧大 → 扣分
    "imdb_buzz_log":         0.27,
}

df["hit_score"] = sum(w * norm(df[f]) for f, w in weights.items())
df.sort_values("hit_score", ascending=False).head(10)

,key,netflix_viewing_hours,netflix_weeks,netflix_year_hint,netflix_title,netflix_clean_title,tmdb_title,tmdb_popularity,tmdb_vote_average,tmdb_vote_count,...,tmdb_number_of_seasons,tmdb_number_of_episodes,tmdb_original_language,tmdb_networks,is_hit,binge_velocity,imdb_rating_shrunk,audience_alignment_gap,imdb_buzz_log,hit_score
92,an_gr_s,64320000,3,2021,Grey's Anatomy,grey s anatomy,Grey's Anatomy,1647.218,8.253,9512.0,...,19.0,419.0,en,ABC,True,2.144000e+07,7.598577,0.653,12.836229,0.804917
661,st_th,2874260000,13,2022,Stranger Things,stranger things,Stranger Things,185.711,8.624,16161.0,...,4.0,34.0,en,Netflix,True,2.210969e+08,6.695653,0.524,4.094345,0.737659
447,he_mo,1170200000,14,2021,Money Heist,money heist,Money Heist,96.354,8.257,17836.0,...,3.0,41.0,es,"Netflix, Antena 3",True,8.358571e+07,8.198638,0.057,13.325201,0.706600
155,bl_pe,185340000,5,2022,Peaky Blinders,peaky blinders,Peaky Blinders,344.477,8.546,8836.0,...,6.0,36.0,en,"BBC One, BBC Two",True,3.706800e+07,8.698615,0.154,13.571087,0.674801
55,ac_um,414630000,5,2022,The Umbrella Academy,the umbrella academy,The Umbrella Academy,51.859,8.604,8891.0,...,4.0,32.0,en,Netflix,True,8.292600e+07,7.797993,0.804,12.663371,0.642177
607,oz,751600000,13,2022,Ozark,ozark,Ozark,68.006,8.244,1968.0,...,4.0,44.0,en,Netflix,True,5.781538e+07,8.397635,0.156,12.886401,0.639819
630,ri_vi,624250000,6,2021,Virgin River,virgin river,Virgin River,54.192,8.032,448.0,...,5.0,54.0,en,Netflix,True,1.040417e+08,7.392821,0.632,11.003782,0.638520
166,bo_to,206170000,3,2022,Top Boy,top boy,Top Boy,57.060,8.133,83.0,...,3.0,24.0,en,Netflix,True,6.872333e+07,8.381152,0.267,10.801818,0.610333
512,ki_la,189570000,5,2022,The Last Kingdom,the last kingdom,The Last Kingdom,143.794,8.290,1485.0,...,5.0,46.0,en,"Netflix, BBC Two",True,3.791400e+07,8.494846,0.210,12.157974,0.595834
102,ar,154270000,6,2021,Arcane,arcane,Arcane,42.447,8.740,3341.0,...,2.0,9.0,en,Netflix,True,2.571167e+07,8.997255,0.260,13.015686,0.590181


## Hit Score 前 25% 的類型、語言與片長分析

In [17]:
top_quartile = df[df["hit_score"] >= df["hit_score"].quantile(0.75)]

print("=== 前 25% 的類型（Genres） ===")
print(top_quartile["imdb_genres"].dropna().str.split(",").explode().str.strip().value_counts())
print()

print("=== 前 25% 的原始語言 ===")
print(top_quartile["tmdb_original_language"].value_counts())
print()

print("=== 前 25% 的片長（分鐘） ===")
runtime = pd.to_numeric(top_quartile["imdb_runtimeMinutes"], errors="coerce")
print(runtime.describe())

=== 前 25% 的類型（Genres） ===
imdb_genres
Drama         40
Crime         18
Action        17
Comedy        11
Romance       11
Mystery        8
Adventure      7
Animation      5
Horror         3
Fantasy        2
Thriller       2
Reality-TV     2
Music          2
History        1
Biography      1
Sci-Fi         1
Name: count, dtype: int64

=== 前 25% 的原始語言 ===
tmdb_original_language
en    43
ko    12
es     5
ja     3
tr     2
pt     2
hi     1
da     1
de     1
Name: count, dtype: int64

=== 前 25% 的片長（分鐘） ===
count    46.000000
mean     46.543478
std      15.395102
min      12.000000
25%      30.000000
50%      45.000000
75%      60.000000
max      84.000000
Name: imdb_runtimeMinutes, dtype: float64
